<a href='https://colab.research.google.com/github/fralfaro/ics294-latex/blob/main/talleres/taller_01_solucion.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

# Solución Taller 1: avance preliminar

Esta es la solución de referencia del Taller 1 con la base `wage1`. En Google Colab, selecciona un entorno de ejecución con kernel **R**. La versión sin respuestas para estudiantes está en `talleres/taller_01_guia.ipynb`.

## Preparación

In [ ]:
paquetes <- c("wooldridge", "dplyr", "ggplot2", "knitr")
instalar <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(instalar) > 0) install.packages(instalar, repos = "https://cloud.r-project.org")
invisible(lapply(paquetes, library, character.only = TRUE))
theme_set(theme_minimal(base_size = 12))
options(width = 110)

data("wage1", package = "wooldridge")
wage_df <- as.data.frame(wage1)
if (!"lwage" %in% names(wage_df)) wage_df$lwage <- log(wage_df$wage)
head(wage_df)

## 1. Conocimiento y estructura de la base

La base contiene 526 observaciones y 24 variables. Para el proyecto se utilizan principalmente `wage`, `lwage`, `educ`, `exper`, `tenure`, `female`, `married` y `nonwhite`. La variable dependiente será `lwage`: el logaritmo reduce la asimetría del salario y permite interpretar los coeficientes de variables en niveles como cambios porcentuales aproximados.

In [ ]:
dim(wage_df)
str(wage_df)

variables_interes <- c("wage", "lwage", "educ", "exper", "tenure",
                       "female", "married", "nonwhite")
tabla_variables <- data.frame(
  variable = variables_interes,
  tipo = vapply(wage_df[variables_interes], function(x) paste(class(x), collapse = ", "), character(1)),
  descripcion_economica = c(
    "salario por hora", "logaritmo del salario por hora", "años de educación",
    "años de experiencia laboral", "años en el puesto actual",
    "dummy: 1 si es mujer", "dummy: 1 si está casado/a",
    "dummy: 1 si es no blanco"
  )
)
tabla_variables

`wage` mide el salario por hora; `educ` representa capital humano general; `exper` captura experiencia laboral potencial y `tenure` capital humano específico del puesto. Las variables binarias permiten describir diferencias entre grupos y, si se justifican, incorporarlas como controles. Esta elección permite estudiar cómo se relacionan educación, experiencia y antigüedad con el salario por hora.

## 2. Auditoría de calidad

La auditoría encuentra 0 valores faltantes, 0 filas incompletas y 0 filas duplicadas. Los rangos son: `wage` entre 0,53 y 24,98; `educ` entre 0 y 18 años; `exper` entre 1 y 51 años; y `tenure` entre 0 y 44 años.

In [ ]:
colSums(is.na(wage_df))
sum(!complete.cases(wage_df))
sum(duplicated(wage_df))

variables_rango <- c("wage", "educ", "exper", "tenure")
lapply(wage_df[variables_rango], range, na.rm = TRUE)

q <- quantile(wage_df$wage, probs = c(.25, .75))
iqr <- q[2] - q[1]
limites_wage <- c(inferior = q[1] - 1.5 * iqr, superior = q[2] + 1.5 * iqr)
atipicos_wage <- subset(wage_df, wage < limites_wage[1] | wage > limites_wage[2])
limites_wage
nrow(atipicos_wage)

El criterio del rango intercuartílico identifica 36 salarios por encima del límite superior descriptivo de 12,205. Esto no prueba que sean errores: los salarios altos son plausibles y no existe una justificación económica para eliminarlos automáticamente. Por ello se conserva la muestra completa y se registra la posible influencia en la interpretación.

## 3. Estadística descriptiva

La siguiente función genera las medidas solicitadas y el segundo bloque resume las variables binarias.

In [ ]:
variables_numericas <- c("wage", "lwage", "educ", "exper", "tenure")

describir <- function(nombre) {
  x <- wage_df[[nombre]]
  data.frame(variable = nombre, n = sum(!is.na(x)), media = mean(x), sd = sd(x),
             minimo = min(x), p25 = quantile(x, .25), mediana = median(x),
             p75 = quantile(x, .75), maximo = max(x))
}
tabla_descriptiva <- do.call(rbind, lapply(variables_numericas, describir))
tabla_descriptiva |> mutate(across(where(is.numeric), ~ round(.x, 3)))

variables_binarias <- c("female", "married", "nonwhite")
tabla_binarias <- lapply(variables_binarias, function(nombre) {
  f <- table(wage_df[[nombre]], useNA = "ifany")
  data.frame(variable = nombre, valor = names(f), frecuencia = as.vector(f),
             porcentaje = round(100 * as.vector(f) / sum(f), 2))
})
do.call(rbind, tabla_binarias)

| Variable | Media | Desv. estándar | Mínimo | P25 | Mediana | P75 | Máximo |
|---|---:|---:|---:|---:|---:|---:|---:|
| `wage` | 5,896 | 3,693 | 0,530 | 3,330 | 4,650 | 6,880 | 24,980 |
| `lwage` | 1,623 | 0,532 | -0,635 | 1,203 | 1,537 | 1,929 | 3,218 |
| `educ` | 12,563 | 2,769 | 0 | 12 | 12 | 14 | 18 |
| `exper` | 17,017 | 13,572 | 1 | 5 | 13,5 | 26 | 51 |
| `tenure` | 5,105 | 7,224 | 0 | 0 | 2 | 7 | 44 |

En las variables binarias: `female = 1` representa 47,91% de la muestra; `married = 1`, 60,84%; y `nonwhite = 1`, 10,27%. El salario tiene media mayor que mediana, señal de asimetría positiva. En cambio, `lwage` es más compacto. `exper` muestra la mayor dispersión absoluta entre las variables de capital humano, mientras que `tenure` también presenta una cola derecha importante.

## 4. Visualización

In [ ]:
ggplot(wage_df, aes(wage)) +
  geom_histogram(bins = 30, fill = "#377eb8", color = "white") +
  labs(title = "Distribución del salario por hora", x = "Salario por hora", y = "Frecuencia")

ggplot(wage_df, aes(lwage)) +
  geom_histogram(bins = 30, fill = "#e41a1c", color = "white") +
  labs(title = "Distribución del log-salario", x = "Logaritmo del salario por hora", y = "Frecuencia")

ggplot(wage_df, aes(x = factor(female), y = lwage)) +
  geom_boxplot() + scale_x_discrete(labels = c("0" = "Hombre", "1" = "Mujer")) +
  labs(title = "Log-salario según sexo", x = NULL, y = "Log-salario")

ggplot(wage_df, aes(x = factor(married), y = lwage)) +
  geom_boxplot() + scale_x_discrete(labels = c("0" = "No casado/a", "1" = "Casado/a")) +
  labs(title = "Log-salario según estado civil", x = NULL, y = "Log-salario")

ggplot(wage_df, aes(educ, lwage)) +
  geom_point(alpha = .45, color = "#377eb8") +
  geom_smooth(method = "lm", se = TRUE, color = "#e41a1c") +
  labs(title = "Educación y log-salario", x = "Años de educación", y = "Log-salario")

ggplot(wage_df, aes(exper, lwage)) +
  geom_point(alpha = .45) + geom_smooth(method = "loess", se = TRUE) +
  labs(title = "Experiencia y log-salario", x = "Años de experiencia", y = "Log-salario")

El histograma de `wage` presenta sesgo a la derecha y valores altos; `lwage` es visualmente más simétrico. Los boxplots muestran un log-salario mediano descriptivamente mayor para hombres que para mujeres y mayor para personas casadas que no casadas. En el gráfico educación-salario se observa una asociación positiva. La relación entre experiencia y log-salario es positiva, aunque parece aplanarse para niveles altos de experiencia, lo que motiva considerar un término cuadrático.

Estas comparaciones son incondicionales: no mantienen constantes educación, experiencia, antigüedad u otras características. Por tanto, son evidencia descriptiva y no una estimación causal.

In [ ]:
variables_correlacion <- c("wage", "lwage", "educ", "exper", "tenure",
                          "female", "married", "nonwhite")
round(cor(wage_df[variables_correlacion]), 3)

ggplot(wage_df, aes(x = exper, y = lwage, color = factor(female))) +
  geom_point(alpha = .45) + geom_smooth(method = "lm", se = FALSE) +
  labs(title = "Experiencia y log-salario por sexo", x = "Experiencia",
       y = "Log-salario", color = "female")

Las correlaciones de `lwage` con `educ`, `exper` y `tenure` son 0,431, 0,111 y 0,326, respectivamente. `exper` y `tenure` tienen correlación 0,499: es una relación relevante para discutir, pero no basta por sí sola para diagnosticar colinealidad problemática. Las correlaciones tampoco son efectos parciales.

## 5. Pregunta de investigación e hipótesis

**Pregunta:** ¿Cómo se relacionan la educación, la experiencia laboral y la antigüedad en el puesto con el logaritmo del salario por hora?

**Hipótesis:**

1. La educación tiene una relación positiva con `lwage`, porque puede aumentar la productividad y las oportunidades laborales.
2. La experiencia tiene una relación positiva, pero posiblemente decreciente, con `lwage`: los primeros años pueden aportar más que los años adicionales posteriores.
3. La antigüedad tiene una relación positiva con `lwage`, porque captura aprendizaje específico y estabilidad en el puesto.
4. Como extensión, se puede estudiar si existe una diferencia condicional por sexo mediante `female`.

Las dos primeras hipótesis pueden observarse descriptivamente de forma individual, pero la pregunta principal requiere considerar varias variables simultáneamente. Una asociación no es necesariamente causal porque educación, experiencia y salario pueden estar relacionados con habilidad no observada, tipo de ocupación, sector, ubicación u otras variables omitidas.

## 6. Modelo propuesto y estimación de referencia

El modelo principal es:

$$lwage_i = \beta_0 + \beta_1 educ_i + \beta_2 exper_i + \beta_3 tenure_i + u_i.$$

Como especificación alternativa se agrega experiencia al cuadrado y una dummy de sexo:

$$lwage_i = \beta_0 + \beta_1 educ_i + \beta_2 exper_i + \beta_3 exper_i^2 + \beta_4 tenure_i + \beta_5 female_i + u_i.$$

In [ ]:
modelo_principal <- lm(lwage ~ educ + exper + tenure, data = wage_df)
modelo_alternativo <- lm(lwage ~ educ + exper + I(exper^2) + tenure + female, data = wage_df)

summary(modelo_principal)
coef(summary(modelo_principal))
summary(modelo_alternativo)

tabla_ajuste <- data.frame(
  modelo = c("principal", "alternativo"),
  R2 = c(summary(modelo_principal)$r.squared, summary(modelo_alternativo)$r.squared),
  R2_ajustado = c(summary(modelo_principal)$adj.r.squared, summary(modelo_alternativo)$adj.r.squared),
  RMSE = c(summary(modelo_principal)$sigma, summary(modelo_alternativo)$sigma)
)
tabla_ajuste |> mutate(across(where(is.numeric), ~ round(.x, 4)))

### Interpretación de los modelos

El modelo principal estimado es aproximadamente:

$$\widehat{lwage}=0{,}2844+0{,}0920\,educ+0{,}0041\,exper+0{,}0221\,tenure.$$

Manteniendo constantes experiencia y antigüedad, un año adicional de educación se asocia aproximadamente con un aumento de 9,20% en el salario; la transformación exacta es $100(e^{0{,}0920}-1)=9,65\%$. Los efectos aproximados de un año adicional de experiencia y antigüedad son 0,41% y 2,21%, respectivamente. Estas son interpretaciones condicionales y descriptivas.

El modelo principal tiene $R^2=0{,}3160$, $R^2$ ajustado igual a 0,3121 y RMSE igual a 0,4409. En la prueba individual bilateral al 5%, `educ`, `exper` y `tenure` son estadísticamente significativas: sus estadísticos $t$ son aproximadamente 12,56, 2,39 y 7,13. La significancia estadística no resuelve por sí sola el problema de causalidad.

En la especificación alternativa, el coeficiente de `exper^2` es aproximadamente -0,000648, consistente con retornos decrecientes de la experiencia. El coeficiente de `female` es aproximadamente -0,298; manteniendo constantes las demás variables, corresponde a una diferencia salarial condicional aproximada de $100(e^{-0{,}298}-1)=-25,8\%$. Esta diferencia no debe interpretarse como discriminación causal sin supuestos y controles adicionales. El $R^2$ de la alternativa es 0,4341, pero el aumento también refleja la incorporación de más variables y debe interpretarse con criterio económico.

## Conclusión

La base no presenta faltantes ni duplicados y permite construir un avance reproducible. El salario en niveles es asimétrico, por lo que `lwage` resulta una transformación útil. La evidencia descriptiva y el modelo propuesto sugieren relaciones positivas de educación, experiencia y antigüedad con el log-salario, además de una posible no linealidad en experiencia.

El siguiente paso es revisar supuestos, inferencia y especificación con mayor detalle en el Taller 2.